In [ ]:
# =============================================================================
# LS-VINE: Learning Vine-Friendly Latent Representations
# Complete Python Implementation for Benchmark and Analysis
#
# Author: Mohsen Ben Hassine
# License: MIT
# =============================================================================

# =============================================================================
# SECTION 1: INSTALLATION AND SETUP
# =============================================================================

# %% 1.1 Install Dependencies
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "pyvinecopulib", "torch", "numpy", "scipy", "matplotlib",
                "seaborn", "scikit-learn", "pandas", "openpyxl"])
print("✅ Installation complete. Runtime → T4 GPU.")


# %% 1.2 Imports and Global Configuration
import warnings
import gc
import time
import traceback
import os
import random
from pathlib import Path
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler

from scipy.stats import kendalltau, rankdata, norm, wilcoxon, friedmanchisquare
from sklearn.decomposition import PCA, FastICA, FactorAnalysis, KernelPCA

warnings.filterwarnings("ignore")

# Check pyvinecopulib availability
try:
    import pyvinecopulib as pv
    VINE_OK = True
    print("✅ pyvinecopulib loaded")
except ImportError:
    VINE_OK = False
    print("❌ WARNING: pyvinecopulib missing")

# Device setup
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print(f"Device: {DEVICE} | AMP: {USE_AMP}")

# Create directories
for d in ["results", "results/figures", "results/tables"]:
    Path(d).mkdir(parents=True, exist_ok=True)


# =============================================================================
# SECTION 2: UTILITY FUNCTIONS
# =============================================================================

# %% 2.1 Seed Management
def set_seed(seed: int = 42):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == "cuda":
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)


# %% 2.2 Memory Monitoring
def _mem_mb():
    """Return current GPU memory usage in MB if available."""
    if DEVICE.type == "cuda":
        return torch.cuda.memory_allocated() / 1e6
    try:
        import resource
        return resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
    except ImportError:
        return 0.0


# =============================================================================
# SECTION 3: CONFIGURATION
# =============================================================================

# %% 3.1 Config Class
@dataclass
class Config:
    """Global hyperparameters for LS-Vine experiments."""

    # Training parameters
    epochs: int = 500
    batch_size: int = 512
    lr: float = 1e-3
    patience: int = 15
    pretrain_epochs: int = 60

    # LS-Vine specific parameters
    gamma: float = 0.01           # Regularization weight
    lam_var: float = 0.002        # Variance regularization weight
    alpha: float = 0.5            # Soft tau loss weight
    beta_tau: float = 1.0         # Temperature for soft Kendall's tau
    K: int = 5                    # Vine re-estimation interval
    tau_w: float = 15.0           # Warmup timescale
    lam_max: float = 1.0          # Maximum vine loss weight
    frac_pairs: float = 0.5       # Fraction of pairs for soft tau
    n_quantiles: int = 20
    tail_weight: float = 2.0
    trunc: int = 3                # Vine truncation depth

    # VAE specific
    vae_beta: float = 0.1

    # WAE / InfoVAE
    wae_lambda: float = 10.0

    # Experiment settings
    n_seeds: int = 10
    seeds: List[int] = field(default_factory=lambda: list(range(42, 52)))
    hidden: int = 256
    d_lat_var_target: float = 0.90
    n_boot_real: int = 20

    # Truncated vine baseline
    trunc_short: int = 1

    # Stability tracking
    compute_stability: bool = False

CFG = Config()


# =============================================================================
# SECTION 4: NEURAL NETWORK MODELS
# =============================================================================

# %% 4.1 MLP Base Class
class MLP(nn.Module):
    """Multi-layer perceptron with ELU activations and batch norm."""

    def __init__(self, d_in: int, d_out: int, hidden: int = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, hidden),
            nn.BatchNorm1d(hidden),
            nn.ELU(),
            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.ELU(),
            nn.Linear(hidden, d_out),
        )

    def forward(self, x):
        return self.net(x)


# %% 4.2 LS-Vine Network
class LSVineNet(nn.Module):
    """LS-Vine encoder-decoder architecture."""

    def __init__(self, d_in: int, d_lat: int, hidden: int = 256):
        super().__init__()
        self.encoder = MLP(d_in, d_lat, hidden)
        self.decoder = MLP(d_lat, d_in, hidden)

    def forward(self, x):
        z = self.encoder(x)
        return z, self.decoder(z)


# %% 4.3 Autoencoder Network
class AENet(nn.Module):
    """Standard autoencoder."""

    def __init__(self, d_in: int, d_lat: int, hidden: int = 256):
        super().__init__()
        self.encoder = MLP(d_in, d_lat, hidden)
        self.decoder = MLP(d_lat, d_in, hidden)

    def forward(self, x):
        z = self.encoder(x)
        return z, self.decoder(z)


# %% 4.4 VAE Encoder and Network
class VAEEncoder(nn.Module):
    """Variational autoencoder encoder with reparameterization."""

    def __init__(self, d_in: int, d_lat: int, hidden: int = 256):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Linear(d_in, hidden),
            nn.BatchNorm1d(hidden),
            nn.ELU(),
            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),
            nn.ELU(),
        )
        self.mu = nn.Linear(hidden, d_lat)
        self.logvar = nn.Linear(hidden, d_lat)

    def forward(self, x):
        h = self.shared(x)
        return self.mu(h), self.logvar(h)


class VAENet(nn.Module):
    """Variational autoencoder."""

    def __init__(self, d_in: int, d_lat: int, hidden: int = 256):
        super().__init__()
        self.encoder = VAEEncoder(d_in, d_lat, hidden)
        self.decoder = MLP(d_lat, d_in, hidden)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x):
        mu, logvar = self.encoder(x)
        z = self.reparameterize(mu, logvar)
        return z, self.decoder(z), mu, logvar


# =============================================================================
# SECTION 5: LOSS FUNCTIONS
# =============================================================================

# %% 5.1 Helper Functions
def _robust_beta_per_dim(X, eps=1e-3):
    """
    Compute dimension-specific temperature parameters for soft Kendall's tau.
    Uses median of pairwise differences for robustness to heavy tails.
    """
    n, d = X.shape
    mask = ~torch.eye(n, dtype=torch.bool, device=X.device)
    betas = []
    for k in range(d):
        diffs = (X[:, k].unsqueeze(0) - X[:, k].unsqueeze(1)).abs()
        med = diffs[mask].median()
        betas.append(1.0 / (med + eps))
    return torch.stack(betas)


def lambda_warmup(t, lam_max=1.0, tau=15.0):
    """Exponential warmup schedule for vine loss weight."""
    return lam_max * (1 - np.exp(-t / tau))


# %% 5.2 Soft Kendall's Tau Reconstruction Loss
def soft_tau_loss(X, X_hat, beta=1.0, frac_pairs=None, gen=None, robust=True):
    """
    Differentiable soft Kendall's tau reconstruction loss.

    Args:
        X: Original data (batch_size, d)
        X_hat: Reconstructed data (batch_size, d)
        beta: Temperature parameter
        frac_pairs: Fraction of dimension pairs to sample
        gen: Random generator for reproducibility
        robust: Use robust per-dimension beta scaling

    Returns:
        Soft tau reconstruction loss (scalar)
    """
    n, d = X.shape

    # Sample dimension pairs
    pairs = [(i, j) for i in range(d) for j in range(i + 1, d)]
    total = len(pairs)
    n_pairs = total if frac_pairs is None else max(10, int(frac_pairs * total))

    if n_pairs < total:
        idx = (torch.randperm(total, generator=gen) if gen else torch.randperm(total))[:n_pairs]
        pairs = [pairs[k] for k in idx.tolist()]
        scale = total / n_pairs
    else:
        scale = 1.0

    # Beta scaling
    beta_vec = _robust_beta_per_dim(X) if robust else torch.full((d,), float(beta), device=X.device)
    mask = torch.triu(torch.ones(n, n, device=X.device, dtype=torch.bool), diagonal=1)

    total_loss = 0.0
    for i, j in pairs:
        bi, bj = beta_vec[i], beta_vec[j]

        # Pairwise differences
        dxi_t = X[:, i].unsqueeze(0) - X[:, i].unsqueeze(1)
        dxj_t = X[:, j].unsqueeze(0) - X[:, j].unsqueeze(1)
        dxi_h = X_hat[:, i].unsqueeze(0) - X_hat[:, i].unsqueeze(1)
        dxj_h = X_hat[:, j].unsqueeze(0) - X_hat[:, j].unsqueeze(1)

        # Soft Kendall's tau
        tau_t = torch.sigmoid((dxi_t * dxj_t)[mask] * bi * bj).mean() * 2 - 1
        tau_h = torch.sigmoid((dxi_h * dxj_h)[mask] * bi * bj).mean() * 2 - 1

        total_loss += (tau_t - tau_h) ** 2

    return scale * total_loss / max(len(pairs), 1)


# %% 5.3 Soft Kendall's Tau Matrix
def soft_kendall_tau_matrix(X, beta=1.0, frac_pairs=None, gen=None, robust=True):
    """Compute the full soft Kendall's tau matrix for a dataset."""
    n, d = X.shape

    pairs = [(i, j) for i in range(d) for j in range(i + 1, d)]
    total = len(pairs)
    n_pairs = total if frac_pairs is None else max(10, int(frac_pairs * total))

    if n_pairs < total:
        idx = (torch.randperm(total, generator=gen) if gen else torch.randperm(total))[:n_pairs]
        pairs = [pairs[k] for k in idx.tolist()]

    beta_vec = _robust_beta_per_dim(X) if robust else torch.full((d,), float(beta), device=X.device)
    mask = ~torch.eye(n, dtype=torch.bool, device=X.device)

    T = torch.eye(d, device=X.device, dtype=X.dtype)
    for i, j in pairs:
        dxi = X[:, i].unsqueeze(0) - X[:, i].unsqueeze(1)
        dxj = X[:, j].unsqueeze(0) - X[:, j].unsqueeze(1)
        T[i, j] = (torch.tanh(beta_vec[i] * dxi) * torch.tanh(beta_vec[j] * dxj))[mask].mean()
        T[j, i] = T[i, j]

    return T


# %% 5.4 Rank-Distribution Matching Loss (L_v)
def rank_dependence_loss(X, Z, beta=1.0, frac_pairs=None, gen=None,
                         n_quantiles: int = 20, tail_weight: float = 2.0):
    """
    Rank-distribution matching loss (L_v).

    Forces the latent space to preserve the distribution of pairwise
    dependence strengths from the observed data.
    """
    Tx = soft_kendall_tau_matrix(X, beta=beta, frac_pairs=frac_pairs, gen=gen)
    Tz = soft_kendall_tau_matrix(Z, beta=beta, frac_pairs=frac_pairs, gen=gen)

    mask_x = ~torch.eye(Tx.shape[0], dtype=torch.bool, device=X.device)
    mask_z = ~torch.eye(Tz.shape[0], dtype=torch.bool, device=Z.device)

    tx_abs, tz_abs = Tx[mask_x].abs(), Tz[mask_z].abs()

    q_lin = torch.linspace(0.0, 1.0, n_quantiles, device=X.device, dtype=tx_abs.dtype)
    q_levels = q_lin ** (1.0 / max(tail_weight, 1e-6))

    return ((torch.quantile(tx_abs, q_levels) - torch.quantile(tz_abs, q_levels)) ** 2).mean()


# %% 5.5 Regularization Loss
def reg_loss(Z, lam_var: float = 0.0):
    """Latent regularization loss penalizing deviations from zero mean and unit variance."""
    mu_pen = (Z.mean(0) ** 2).sum()
    if lam_var <= 0:
        return mu_pen
    log_std = torch.log(Z.std(0) + 1e-6)
    return mu_pen + lam_var * (log_std ** 2).sum()


# %% 5.6 MMD Penalty for WAE/InfoVAE
def mmd_penalty(z, lam=10.0, sigma=1.0):
    """RBF MMD against N(0,1). Returns a differentiable tensor."""
    if z.dim() == 1:
        z = z.unsqueeze(0)
    z_prior = torch.randn_like(z)
    n = z.shape[0]

    k_xx = torch.exp(-torch.cdist(z, z) ** 2 / (2 * sigma ** 2))
    k_yy = torch.exp(-torch.cdist(z_prior, z_prior) ** 2 / (2 * sigma ** 2))
    k_xy = torch.exp(-torch.cdist(z, z_prior) ** 2 / (2 * sigma ** 2))

    mmd = (k_xx.sum() - k_xx.diag().sum()) / (n * (n - 1) + 1e-8) + \
          (k_yy.sum() - k_yy.diag().sum()) / (n * (n - 1) + 1e-8) - \
          2 * k_xy.mean()

    return lam * torch.clamp(mmd, min=0.0)


# =============================================================================
# SECTION 6: VINE UTILITIES
# =============================================================================

# %% 6.1 Empirical PIT
def empirical_pit(X):
    """Empirical Probability Integral Transform (rank-based)."""
    n, d = X.shape
    return np.column_stack([rankdata(X[:, j]) / (n + 1) for j in range(d)])


# %% 6.2 Kendall's Tau Matrix
def kendall_matrix(X: np.ndarray) -> np.ndarray:
    """Compute the full Kendall's tau matrix for a dataset."""
    d = X.shape[1]
    return np.array([[kendalltau(X[:, i], X[:, j])[0] for j in range(d)] for i in range(d)])


# %% 6.3 Vine Fitting
def fit_vine(U: np.ndarray, trunc_lvl: int = 3):
    """Fit a vine copula using AIC selection over 5 families."""
    if not VINE_OK:
        raise RuntimeError("pyvinecopulib is not available")

    U_safe = np.clip(U, 1e-5, 1.0 - 1e-5)
    ctrl = pv.FitControlsVinecop(
        family_set=[pv.BicopFamily.gaussian, pv.BicopFamily.student,
                    pv.BicopFamily.clayton, pv.BicopFamily.gumbel,
                    pv.BicopFamily.frank],
        trunc_lvl=trunc_lvl,
        selection_criterion="aic"
    )
    return pv.Vinecop.from_data(data=U_safe, controls=ctrl)


def fit_vine_robust(U, trunc_lvl=3, verbose=False):
    """Robust vine fitting with fallback on simpler models."""
    if not VINE_OK or U is None or len(U) == 0 or U.shape[1] < 2:
        return None

    U_safe = np.clip(U, 1e-5, 1.0 - 1e-5)

    # Remove constant columns
    var = np.var(U_safe, axis=0)
    if (var < 1e-8).any():
        U_safe = U_safe[:, var > 1e-8]
        if U_safe.shape[1] < 2:
            return None

    try:
        return fit_vine(U_safe, trunc_lvl)
    except Exception:
        # Fallback: trunc_lvl=1 with Gaussian + Student
        try:
            ctrl = pv.FitControlsVinecop(
                family_set=[pv.BicopFamily.gaussian, pv.BicopFamily.student],
                trunc_lvl=1,
                selection_criterion="aic"
            )
            return pv.Vinecop.from_data(data=U_safe, controls=ctrl)
        except Exception:
            return None


# %% 6.4 Vine Metrics
def vine_metrics(vine, U: np.ndarray):
    """Compute log-likelihood and AIC for a vine model."""
    if U is None or len(U) == 0:
        return np.nan, np.inf

    U_safe = np.clip(U, 1e-5, 1.0 - 1e-5)
    try:
        ll = vine.loglik(U_safe) / len(U_safe)
        if np.isnan(ll):
            return np.nan, np.inf
        aic = -2 * vine.loglik(U_safe) + 2 * vine.npars
        return ll, aic
    except Exception:
        return np.nan, np.inf


def vine_bic(vine, U: np.ndarray) -> float:
    """Compute BIC for a vine model."""
    if U is None or len(U) == 0:
        return np.inf
    U_safe = np.clip(U, 1e-5, 1.0 - 1e-5)
    try:
        bic = -2 * vine.loglik(U_safe) + vine.npars * np.log(len(U_safe))
        return np.nan if np.isnan(bic) else bic
    except Exception:
        return np.inf


def vine_nparams(vine) -> float:
    """Get the number of parameters in a vine model."""
    return vine.npars


# =============================================================================
# SECTION 7: DATA GENERATORS
# =============================================================================

# %% 7.1 Student-t D-Vine
def make_student_dvine(d, rho, nu, n, seed=77):
    """Generate Student-t D-vine data."""
    if not VINE_OK:
        raise RuntimeError("pyvinecopulib is missing, cannot generate student D-vine.")

    pc = []
    for tree in range(d - 1):
        pc.append([pv.Bicop(family=pv.BicopFamily.student,
                            parameters=np.array([[rho], [float(nu)]]))
                   for _ in range(d - 1 - tree)])

    vine = pv.Vinecop.from_structure(
        structure=pv.DVineStructure(list(range(1, d + 1))),
        pair_copulas=pc
    )
    U = vine.simulate(n, seeds=[seed])
    return norm.ppf(np.clip(U, 1e-6, 1 - 1e-6))


# %% 7.2 Mixed R-Vine
def make_mixed_rvine(d, n, seed=42):
    """
    Generate mixed R-vine data with heterogeneous dependencies.

    Steps:
    1. Generate Gaussian data with block correlation structure
    2. Fit a true R-vine with mixed families
    3. Sample from the fitted R-vine
    """
    if not VINE_OK:
        raise RuntimeError("pyvinecopulib is required for mixed R-vine generation")

    np.random.seed(seed)

    # Step 1: Gaussian data with block correlations
    Sigma = np.eye(d)
    block_size = max(1, d // 3)

    # Block 1: Strong correlation
    for i in range(block_size):
        for j in range(block_size):
            if i != j:
                Sigma[i, j] = 0.7

    # Block 2: Moderate correlation
    for i in range(block_size, 2 * block_size):
        for j in range(block_size, 2 * block_size):
            if i != j:
                Sigma[i, j] = 0.5

    # Block 3: Weak correlation
    for i in range(2 * block_size, d):
        for j in range(2 * block_size, d):
            if i != j:
                Sigma[i, j] = 0.3

    # Cross-block correlations
    for i in range(d):
        for j in range(d):
            if Sigma[i, j] == 0 and i != j:
                Sigma[i, j] = 0.15

    L = np.linalg.cholesky(Sigma)
    Z = np.random.randn(n, d) @ L.T
    U = norm.cdf(Z)

    # Step 2: Fit R-vine with mixed families
    U_safe = np.clip(U, 1e-5, 1.0 - 1e-5)
    controls = pv.FitControlsVinecop(
        family_set=[pv.BicopFamily.student, pv.BicopFamily.clayton,
                    pv.BicopFamily.gumbel, pv.BicopFamily.frank],
        trunc_lvl=min(3, d - 1),
        selection_criterion="aic"
    )
    vine = pv.Vinecop.from_data(data=U_safe, controls=controls)

    # Step 3: Sample from fitted R-vine
    U_sim = vine.simulate(n, seeds=[seed])
    return norm.ppf(np.clip(U_sim, 1e-6, 1 - 1e-6))


# %% 7.3 S&P500-Calibrated Data
def make_sp500_calibrated(n=3000, d=20, seed=42):
    """Generate synthetic S&P500-calibrated financial returns."""
    np.random.seed(seed)
    from scipy.stats import t as t_dist

    s = 4
    sectors = d // s

    Sigma = np.full((d, d), 0.25)
    for i in range(sectors):
        Sigma[i * s:(i + 1) * s, i * s:(i + 1) * s] = 0.65
    np.fill_diagonal(Sigma, 1.0)

    L = np.linalg.cholesky(Sigma)
    X = t_dist.rvs(df=4.5, size=(n, d)) @ L.T

    vols = np.random.uniform(0.20, 0.35, d) / np.sqrt(252)
    X *= vols

    # Add localized market shocks
    X[500:520] *= 3.5
    X[500:510] -= 0.025

    return X


# %% 7.4 ERA5-Calibrated Data
def make_era5_calibrated(n=3000, d=15, seed=43):
    """Generate synthetic ERA5-calibrated meteorological data."""
    np.random.seed(seed)
    from scipy.stats import t as t_dist

    c = 5
    n_cl = d // c

    Sigma = np.full((d, d), 0.18)
    for i in range(n_cl):
        Sigma[i * c:(i + 1) * c, i * c:(i + 1) * c] = 0.72
    np.fill_diagonal(Sigma, 1.0)

    L = np.linalg.cholesky(Sigma)
    X = t_dist.rvs(df=5.0, size=(n, d)) @ L.T

    # Add seasonal patterns
    t_idx = np.arange(n)
    for c_i in range(n_cl):
        X[:, c_i * c:(c_i + 1) * c] += (0.3 + 0.1 * c_i) * np.sin(2 * np.pi * t_idx / 365)[:, None]

    # Seasonal damping
    X[1000:1365] *= 0.4

    return X


# %% 7.5 Block Factor Student-t (S7)
def make_block_factor_student_t(d=20, n_blocks=4, n_per_block=5,
                                 rho=0.7, nu=4, n_samples=2000, seed=42):
    """
    Generate block-factor Student-t data without localized shocks (S7).
    Used to isolate the effect of localized market shocks on LS-Vine's performance.
    """
    np.random.seed(seed)
    n_factors = n_blocks

    factors = t.rvs(df=nu, size=(n_samples, n_factors))
    noise = t.rvs(df=nu, size=(n_samples, d))

    X = np.zeros((n_samples, d))
    for b in range(n_blocks):
        start = b * n_per_block
        end = start + n_per_block
        X[:, start:end] = rho * factors[:, b:b + 1] + np.sqrt(1 - rho**2) * noise[:, start:end]

    return X


# %% 7.6 Data Splitting
def split(X, n_tr=2000, n_vl=500):
    """Split data into train, validation, and test sets."""
    return {
        "X_train": X[:n_tr],
        "X_val": X[n_tr:n_tr + n_vl],
        "X_test": X[n_tr + n_vl:],
    }


def split_real(X, frac=(0.6, 0.2, 0.2)):
    """Split real data with specified fractions."""
    n = len(X)
    n1 = int(n * frac[0])
    n2 = int(n * (frac[0] + frac[1]))
    return {
        "X_train": X[:n1],
        "X_val": X[n1:n2],
        "X_test": X[n2:],
    }


# =============================================================================
# SECTION 8: TRAINING LOOPS
# =============================================================================

# %% 8.1 LS-Vine Training
def train_lsvine(X_tr, X_vl, d_lat, cfg=CFG, verbose=True, seed=42):
    """
    Train LS-Vine model with alternating optimization.

    Args:
        X_tr: Training data (n_train, d)
        X_vl: Validation data (n_val, d)
        d_lat: Latent dimension
        cfg: Configuration object
        verbose: Print progress
        seed: Random seed

    Returns:
        model: Trained LSVineNet
        vine: Fitted vine copula
        hist: Training history
        best_mu: Best latent mean
        best_std: Best latent std
    """
    set_seed(seed)
    gen = torch.Generator(device="cpu")
    gen.manual_seed(seed)

    d = X_tr.shape[1]
    model = LSVineNet(d, d_lat, cfg.hidden).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs, eta_min=1e-5)
    scaler = GradScaler() if USE_AMP else None

    Xtr = torch.tensor(X_tr, dtype=torch.float32).to(DEVICE)
    Xvl = torch.tensor(X_vl, dtype=torch.float32).to(DEVICE)

    hist = {"epoch": [], "train_loss": [], "val_aic": [], "lam": [], "lv_train": []}
    curr_vine = None
    best_vine = None
    best_aic = np.inf
    best_state = None
    best_mu = None
    best_std = None
    patience = 0
    global_mu = None
    global_std = None

    try:
        for ep in range(1, cfg.epochs + 1):
            in_pre = ep <= cfg.pretrain_epochs
            lam = 0.0 if in_pre else lambda_warmup(ep - cfg.pretrain_epochs, cfg.lam_max, cfg.tau_w)

            model.train()
            ep_loss = 0.0
            ep_lv = 0.0
            n_lv = 0
            perm = torch.randperm(len(Xtr))

            for i in range(0, len(Xtr), cfg.batch_size):
                xb = Xtr[perm[i:i + cfg.batch_size]]

                with autocast(enabled=USE_AMP):
                    z, xhat = model(xb)
                    Lr = F.mse_loss(xhat, xb) + cfg.alpha * soft_tau_loss(
                        xb, xhat, cfg.beta_tau, cfg.frac_pairs, gen
                    )
                    Lg = reg_loss(z, cfg.lam_var)

                    Lv = torch.zeros((), device=DEVICE, dtype=torch.float32)
                    if lam > 0 and not in_pre:
                        Lv = rank_dependence_loss(
                            xb.float(), z.float(),
                            beta=cfg.beta_tau,
                            frac_pairs=cfg.frac_pairs,
                            gen=gen,
                            n_quantiles=cfg.n_quantiles,
                            tail_weight=cfg.tail_weight,
                        )
                        if torch.isfinite(Lv):
                            ep_lv += float(Lv)
                            n_lv += 1
                        else:
                            Lv = torch.zeros((), device=DEVICE, dtype=torch.float32)

                loss = Lr + lam * Lv + cfg.gamma * Lg

                if not torch.isfinite(loss):
                    opt.zero_grad()
                    continue

                opt.zero_grad()
                if USE_AMP:
                    scaler.scale(loss).backward()
                    scaler.unscale_(opt)
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(opt)
                    scaler.update()
                else:
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    opt.step()

                ep_loss += float(loss)

            sched.step()

            # Periodic vine re-estimation
            if ep % cfg.K == 0 and VINE_OK and not in_pre:
                model.eval()
                with torch.no_grad():
                    Zf, _ = model(Xtr)
                    Zf = Zf.float()
                    global_mu = Zf.mean(0, keepdim=True)
                    global_std = Zf.std(0, keepdim=True)
                    Uf = empirical_pit(Zf.cpu().numpy()).astype(np.float64)
                curr_vine = fit_vine_robust(Uf, cfg.trunc)

            # Validation
            model.eval()
            val_aic = np.nan
            if curr_vine is not None:
                with torch.no_grad():
                    Zv, _ = model(Xvl)
                    Uv = empirical_pit(Zv.float().cpu().numpy()).astype(np.float64)
                _, val_aic = vine_metrics(curr_vine, Uv)

            hist["epoch"].append(ep)
            hist["train_loss"].append(ep_loss)
            hist["val_aic"].append(val_aic)
            hist["lam"].append(lam)
            hist["lv_train"].append(ep_lv / n_lv if n_lv else np.nan)

            if np.isfinite(val_aic) and val_aic < best_aic:
                best_aic = val_aic
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_vine = curr_vine
                best_mu = global_mu.clone() if global_mu is not None else None
                best_std = global_std.clone() if global_std is not None else None
                patience = 0
            else:
                patience += 1

            if patience >= cfg.patience:
                if verbose:
                    print(f"  Early stop ep {ep} (best AIC={best_aic:.2f})")
                break

            if verbose and ep % 50 == 0:
                print(f"  ep {ep:4d} | val_aic={val_aic:.2f} | lam={lam:.3f} | Lv_train={hist['lv_train'][-1]:.4f}")

        if best_state:
            model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    finally:
        del Xtr, Xvl
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return model, best_vine, hist, best_mu, best_std


# %% 8.2 Autoencoder Training
def train_ae(X_tr, X_vl, d_lat, cfg=CFG, seed=42):
    """Train standard autoencoder."""
    set_seed(seed)
    model = AENet(X_tr.shape[1], d_lat, cfg.hidden).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs, eta_min=1e-5)

    Xtr = torch.tensor(X_tr, dtype=torch.float32).to(DEVICE)
    Xvl = torch.tensor(X_vl, dtype=torch.float32).to(DEVICE)

    best_loss, best_state, patience = np.inf, None, 0

    try:
        for ep in range(1, cfg.epochs + 1):
            model.train()
            for i in range(0, len(Xtr), cfg.batch_size):
                xb = Xtr[torch.randperm(len(Xtr))[i:i + cfg.batch_size]]
                z, xhat = model(xb)
                loss = F.mse_loss(xhat, xb)
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            sched.step()
            model.eval()
            with torch.no_grad():
                _, xhat_v = model(Xvl)
                val_loss = float(F.mse_loss(xhat_v, Xvl))

            if val_loss < best_loss:
                best_loss = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1

            if patience >= cfg.patience:
                break

        if best_state:
            model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    finally:
        del Xtr, Xvl
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return model


# %% 8.3 VAE Training
def train_vae(X_tr, X_vl, d_lat, cfg=CFG, seed=42):
    """Train Variational Autoencoder."""
    set_seed(seed)
    model = VAENet(X_tr.shape[1], d_lat, cfg.hidden).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs, eta_min=1e-5)

    Xtr = torch.tensor(X_tr, dtype=torch.float32).to(DEVICE)
    Xvl = torch.tensor(X_vl, dtype=torch.float32).to(DEVICE)

    best_loss, best_state, patience = np.inf, None, 0

    try:
        for ep in range(1, cfg.epochs + 1):
            model.train()
            for i in range(0, len(Xtr), cfg.batch_size):
                xb = Xtr[torch.randperm(len(Xtr))[i:i + cfg.batch_size]]
                z, xhat, mu, lv = model(xb)
                kl_loss = -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())
                loss = F.mse_loss(xhat, xb) + cfg.vae_beta * kl_loss
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            sched.step()
            model.eval()
            with torch.no_grad():
                _, xv, mv, lv_v = model(Xvl)
                val_loss = float(F.mse_loss(xv, Xvl) + cfg.vae_beta * (-0.5 * torch.mean(1 + lv_v - mv.pow(2) - lv_v.exp())))

            if val_loss < best_loss:
                best_loss = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1

            if patience >= cfg.patience:
                break

        if best_state:
            model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    finally:
        del Xtr, Xvl
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return model


# %% 8.4 WAE Training
def train_wae(X_tr, X_vl, d_lat, cfg=CFG, seed=42):
    """Train Wasserstein Autoencoder."""
    set_seed(seed)
    model = AENet(X_tr.shape[1], d_lat, cfg.hidden).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs, eta_min=1e-5)

    Xtr = torch.tensor(X_tr, dtype=torch.float32).to(DEVICE)
    Xvl = torch.tensor(X_vl, dtype=torch.float32).to(DEVICE)

    best_loss, best_state, patience = np.inf, None, 0

    try:
        for ep in range(1, cfg.epochs + 1):
            model.train()
            for i in range(0, len(Xtr), cfg.batch_size):
                xb = Xtr[torch.randperm(len(Xtr))[i:i + cfg.batch_size]]
                z, xhat = model(xb)
                loss = F.mse_loss(xhat, xb) + mmd_penalty(z, lam=cfg.wae_lambda)
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            sched.step()
            model.eval()
            with torch.no_grad():
                z_v, xhat_v = model(Xvl)
                val_loss = float(F.mse_loss(xhat_v, Xvl) + mmd_penalty(z_v, lam=cfg.wae_lambda))

            if val_loss < best_loss:
                best_loss = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1

            if patience >= cfg.patience:
                break

        if best_state:
            model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    finally:
        del Xtr, Xvl
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return model


# %% 8.5 InfoVAE Training
def train_infovae(X_tr, X_vl, d_lat, cfg=CFG, seed=42):
    """Train InfoVAE."""
    set_seed(seed)
    model = VAENet(X_tr.shape[1], d_lat, cfg.hidden).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs, eta_min=1e-5)

    Xtr = torch.tensor(X_tr, dtype=torch.float32).to(DEVICE)
    Xvl = torch.tensor(X_vl, dtype=torch.float32).to(DEVICE)

    best_loss, best_state, patience = np.inf, None, 0

    try:
        for ep in range(1, cfg.epochs + 1):
            model.train()
            for i in range(0, len(Xtr), cfg.batch_size):
                xb = Xtr[torch.randperm(len(Xtr))[i:i + cfg.batch_size]]
                z, xhat, mu, lv = model(xb)
                rec_loss = F.mse_loss(xhat, xb)
                kl_loss = -0.5 * torch.mean(1 + lv - mu.pow(2) - lv.exp())
                mmd_loss = mmd_penalty(z, lam=cfg.wae_lambda)
                loss = rec_loss + mmd_loss + cfg.vae_beta * kl_loss
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            sched.step()
            model.eval()
            with torch.no_grad():
                _, xv, mv, lv_v = model(Xvl)
                val_loss = float(F.mse_loss(xv, Xvl) +
                                  mmd_penalty(mv, lam=cfg.wae_lambda) +
                                  cfg.vae_beta * (-0.5 * torch.mean(1 + lv_v - mv.pow(2) - lv_v.exp())))

            if val_loss < best_loss:
                best_loss = val_loss
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                patience = 0
            else:
                patience += 1

            if patience >= cfg.patience:
                break

        if best_state:
            model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    finally:
        del Xtr, Xvl
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return model


# %% 8.6 AE-Selected Training
def train_ae_selected(X_tr, X_vl, d_lat, cfg=CFG, seed=42, verbose=False):
    """Train autoencoder with soft tau loss and vine selection."""
    set_seed(seed)
    model = AENet(X_tr.shape[1], d_lat, cfg.hidden).to(DEVICE)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs, eta_min=1e-5)
    gen = torch.Generator(device="cpu")
    gen.manual_seed(seed)

    Xtr = torch.tensor(X_tr, dtype=torch.float32).to(DEVICE)
    Xvl = torch.tensor(X_vl, dtype=torch.float32).to(DEVICE)

    curr_vine = None
    best_vine = None
    best_aic = np.inf
    best_state = None
    patience = 0
    global_mu = None
    global_std = None

    try:
        for ep in range(1, cfg.epochs + 1):
            model.train()
            for i in range(0, len(Xtr), cfg.batch_size):
                xb = Xtr[torch.randperm(len(Xtr))[i:i + cfg.batch_size]]
                z, xhat = model(xb)
                loss = F.mse_loss(xhat, xb) + cfg.alpha * soft_tau_loss(
                    xb, xhat, cfg.beta_tau, cfg.frac_pairs, gen
                )
                opt.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()

            sched.step()

            if ep % cfg.K == 0 and VINE_OK and ep > cfg.pretrain_epochs:
                model.eval()
                with torch.no_grad():
                    Zf, _ = model(Xtr)
                    Zf = Zf.float()
                    global_mu = Zf.mean(0, keepdim=True)
                    global_std = Zf.std(0, keepdim=True)
                    Uf = empirical_pit(Zf.cpu().numpy()).astype(np.float64)
                curr_vine = fit_vine_robust(Uf, cfg.trunc)

            model.eval()
            val_aic = np.nan
            if curr_vine is not None:
                with torch.no_grad():
                    Zv, _ = model(Xvl)
                    Uv = empirical_pit(Zv.float().cpu().numpy()).astype(np.float64)
                _, val_aic = vine_metrics(curr_vine, Uv)

            if np.isfinite(val_aic) and val_aic < best_aic:
                best_aic = val_aic
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_vine = curr_vine
                patience = 0
            else:
                patience += 1

            if patience >= cfg.patience:
                if verbose:
                    print(f"  [AE-Vine-Selected] Early stop ep {ep} (best AIC={best_aic:.2f})")
                break

        if best_state:
            model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    finally:
        del Xtr, Xvl
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    return model, best_vine


# =============================================================================
# SECTION 9: BENCHMARK EXECUTION
# =============================================================================

# %% 9.1 Method Definitions
METHODS = [
    "LS-Vine",
    "AE-Vine-Selected",
    "PCA-Vine",
    "ICA-Vine",
    "FA-Vine",
    "KPCA-Vine",
    "AE-Vine",
    "VAE-Vine",
    "WAE-Vine",
    "InfoVAE-Vine",
    "Vine-Direct",
    "Vine-Truncated"
]

SCENARIOS = {
    "S3": dict(label="S3 — Mixed R-vine (d=12, heterogeneous)",
               builder=lambda s: make_mixed_rvine(12, 3500, s),
               split_fn=split, d_lat=4),
    "S1": dict(label="S1 — Student-t D-vine (d=10, ν=4)",
               builder=lambda s: make_student_dvine(10, 0.4, 4, 3500, s),
               split_fn=split, d_lat=4),
    "S2": dict(label="S2 — Student-t D-vine (d=20, ν=4)",
               builder=lambda s: make_student_dvine(20, 0.4, 4, 3500, s),
               split_fn=split, d_lat=5),
    "R1": dict(label="R1 — S&P500-calibrated (d=20, n=3000)",
               builder=lambda s: make_sp500_calibrated(3000, 20, s),
               split_fn=split_real, d_lat=5),
    "R2": dict(label="R2 — ERA5-calibrated (d=15, n=3000)",
               builder=lambda s: make_era5_calibrated(3000, 15, s),
               split_fn=split_real, d_lat=4),
}


# %% 9.2 Latent Dimension Selection
def select_d_lat(X_train, var_threshold=0.90, d_min=2, d_max=None):
    """Select latent dimension based on PCA variance threshold."""
    d_max = d_max or max(d_min, X_train.shape[1] // 2)
    n_comp = min(d_max, X_train.shape[1], X_train.shape[0])
    pca = PCA(n_components=n_comp).fit(X_train)
    cum = np.cumsum(pca.explained_variance_ratio_)
    d_lat = int(np.searchsorted(cum, var_threshold) + 1)
    return int(np.clip(d_lat, d_min, d_max))


# %% 9.3 Method Runner
def run_method(name: str, X_tr, X_vl, X_ts, d_lat, cfg=CFG, seed=42):
    """Run a single method and return results."""
    result = dict(
        method=name, d_lat=d_lat, seed=seed,
        ll=np.nan, nll=np.nan, aic=np.nan, bic=np.nan,
        n_params=np.nan, t_train=np.nan, t_sample=np.nan,
        mem_mb=np.nan, rmse_corr=np.nan, tau_err=np.nan,
        dep_rec=np.nan, stability=np.nan
    )

    d = X_tr.shape[1]
    U_ts_emp = empirical_pit(X_ts)
    tau_orig = kendall_matrix(X_ts)
    corr_orig = np.corrcoef(X_ts.T)

    t0 = time.perf_counter()
    mem0 = _mem_mb()

    try:
        # --- LS-Vine ---
        if name == "LS-Vine":
            model, vine, hist, gmu, gstd = train_lsvine(X_tr, X_vl, d_lat, cfg, verbose=False, seed=seed)
            result["t_train"] = time.perf_counter() - t0
            if vine is None:
                return result

            Xts_t = torch.tensor(X_ts, dtype=torch.float32).to(DEVICE)
            model.eval()
            with torch.no_grad():
                Zts, Xhat = model(Xts_t)
                Uts = empirical_pit(Zts.float().cpu().numpy()).astype(np.float64)
                Xhat_np = Xhat.cpu().numpy()

            ll, aic = vine_metrics(vine, Uts)
            bic = vine_bic(vine, Uts)
            n_prm = vine_nparams(vine)

            ts0 = time.perf_counter()
            try:
                vine.simulate(500, seeds=[seed])
                result["t_sample"] = time.perf_counter() - ts0
            except Exception:
                pass

            tau_rec = kendall_matrix(Xhat_np)
            corr_rec = np.corrcoef(Xhat_np.T)
            result.update(
                ll=ll, nll=-ll, aic=aic, bic=bic, n_params=n_prm,
                dep_rec=np.linalg.norm(tau_orig - tau_rec, "fro"),
                tau_err=np.abs(tau_orig - tau_rec)[np.triu_indices(d, 1)].mean(),
                rmse_corr=np.sqrt(np.mean((corr_orig - corr_rec) ** 2))
            )
            del model

        # --- PCA / ICA / FA-Vine ---
        elif name in ["PCA-Vine", "ICA-Vine", "FA-Vine"]:
            if name == "PCA-Vine":
                proj = PCA(n_components=d_lat, random_state=seed).fit(X_tr)
            elif name == "ICA-Vine":
                proj = FastICA(n_components=d_lat, random_state=seed, whiten='unit-variance').fit(X_tr)
            else:
                proj = FactorAnalysis(n_components=d_lat, random_state=seed).fit(X_tr)

            Ztr, Zts = proj.transform(X_tr), proj.transform(X_ts)
            Utr = empirical_pit(Ztr).astype(np.float64)
            Uts = empirical_pit(Zts).astype(np.float64)

            vine = fit_vine(Utr, cfg.trunc)
            result["t_train"] = time.perf_counter() - t0

            ll, aic = vine_metrics(vine, Uts)
            bic = vine_bic(vine, Uts)

            ts0 = time.perf_counter()
            try:
                vine.simulate(500, seeds=[seed])
                result["t_sample"] = time.perf_counter() - ts0
            except Exception:
                pass

            Xhat = proj.inverse_transform(Zts) if name != "FA-Vine" else Zts @ proj.components_ + proj.mean_
            tau_rec = kendall_matrix(Xhat)
            corr_rec = np.corrcoef(Xhat.T)
            result.update(
                ll=ll, nll=-ll, aic=aic, bic=bic, n_params=vine_nparams(vine),
                dep_rec=np.linalg.norm(tau_orig - tau_rec, "fro"),
                tau_err=np.abs(tau_orig - tau_rec)[np.triu_indices(d, 1)].mean(),
                rmse_corr=np.sqrt(np.mean((corr_orig - corr_rec) ** 2))
            )

        # --- KPCA-Vine ---
        elif name == "KPCA-Vine":
            kpca = KernelPCA(
                n_components=d_lat, kernel='rbf', gamma=1.0/d,
                fit_inverse_transform=True, random_state=seed
            ).fit(X_tr)

            Ztr, Zts = kpca.transform(X_tr), kpca.transform(X_ts)
            Utr = empirical_pit(Ztr).astype(np.float64)
            Uts = empirical_pit(Zts).astype(np.float64)

            vine = fit_vine(Utr, cfg.trunc)
            result["t_train"] = time.perf_counter() - t0

            ll, aic = vine_metrics(vine, Uts)
            bic = vine_bic(vine, Uts)

            ts0 = time.perf_counter()
            try:
                vine.simulate(500, seeds=[seed])
                result["t_sample"] = time.perf_counter() - ts0
            except Exception:
                pass

            Xhat = kpca.inverse_transform(Zts)
            tau_rec = kendall_matrix(Xhat)
            corr_rec = np.corrcoef(Xhat.T)
            result.update(
                ll=ll, nll=-ll, aic=aic, bic=bic, n_params=vine_nparams(vine),
                dep_rec=np.linalg.norm(tau_orig - tau_rec, "fro"),
                tau_err=np.abs(tau_orig - tau_rec)[np.triu_indices(d, 1)].mean(),
                rmse_corr=np.sqrt(np.mean((corr_orig - corr_rec) ** 2))
            )

        # --- AE-Vine ---
        elif name == "AE-Vine":
            model = train_ae(X_tr, X_vl, d_lat, cfg, seed)
            result["t_train"] = time.perf_counter() - t0

            model.eval()
            Xtr_t = torch.tensor(X_tr, dtype=torch.float32).to(DEVICE)
            Xts_t = torch.tensor(X_ts, dtype=torch.float32).to(DEVICE)

            with torch.no_grad():
                Ztr_t, _ = model(Xtr_t)
                Zts_t, Xhat_t = model(Xts_t)
                Utr = empirical_pit(Ztr_t.float().cpu().numpy()).astype(np.float64)
                Uts = empirical_pit(Zts_t.float().cpu().numpy()).astype(np.float64)
                Xhat_np = Xhat_t.float().cpu().numpy()

            vine = fit_vine(Utr, cfg.trunc)
            ll, aic = vine_metrics(vine, Uts)
            bic = vine_bic(vine, Uts)

            ts0 = time.perf_counter()
            try:
                vine.simulate(500, seeds=[seed])
                result["t_sample"] = time.perf_counter() - ts0
            except Exception:
                pass

            tau_rec = kendall_matrix(Xhat_np)
            corr_rec = np.corrcoef(Xhat_np.T)
            result.update(
                ll=ll, nll=-ll, aic=aic, bic=bic, n_params=vine_nparams(vine),
                dep_rec=np.linalg.norm(tau_orig - tau_rec, "fro"),
                tau_err=np.abs(tau_orig - tau_rec)[np.triu_indices(d, 1)].mean(),
                rmse_corr=np.sqrt(np.mean((corr_orig - corr_rec) ** 2))
            )
            del model

        # --- VAE-Vine ---
        elif name == "VAE-Vine":
            model = train_vae(X_tr, X_vl, d_lat, cfg, seed)
            result["t_train"] = time.perf_counter() - t0

            model.eval()
            Xtr_t = torch.tensor(X_tr, dtype=torch.float32).to(DEVICE)
            Xts_t = torch.tensor(X_ts, dtype=torch.float32).to(DEVICE)

            with torch.no_grad():
                _, _, mu_tr, _ = model(Xtr_t)
                _, _, mu_ts, _ = model(Xts_t)
                Xhat_t = model.decoder(mu_ts)
                Utr = empirical_pit(mu_tr.float().cpu().numpy()).astype(np.float64)
                Uts = empirical_pit(mu_ts.float().cpu().numpy()).astype(np.float64)
                Xhat_np = Xhat_t.float().cpu().numpy()

            vine = fit_vine(Utr, cfg.trunc)
            ll, aic = vine_metrics(vine, Uts)
            bic = vine_bic(vine, Uts)

            ts0 = time.perf_counter()
            try:
                vine.simulate(500, seeds=[seed])
                result["t_sample"] = time.perf_counter() - ts0
            except Exception:
                pass

            tau_rec = kendall_matrix(Xhat_np)
            corr_rec = np.corrcoef(Xhat_np.T)
            result.update(
                ll=ll, nll=-ll, aic=aic, bic=bic, n_params=vine_nparams(vine),
                dep_rec=np.linalg.norm(tau_orig - tau_rec, "fro"),
                tau_err=np.abs(tau_orig - tau_rec)[np.triu_indices(d, 1)].mean(),
                rmse_corr=np.sqrt(np.mean((corr_orig - corr_rec) ** 2))
            )
            del model

        # --- WAE-Vine ---
        elif name == "WAE-Vine":
            model = train_wae(X_tr, X_vl, d_lat, cfg, seed)
            result["t_train"] = time.perf_counter() - t0

            model.eval()
            Xtr_t = torch.tensor(X_tr, dtype=torch.float32).to(DEVICE)
            Xts_t = torch.tensor(X_ts, dtype=torch.float32).to(DEVICE)

            with torch.no_grad():
                Ztr_t, _ = model(Xtr_t)
                Zts_t, Xhat_t = model(Xts_t)
                Utr = empirical_pit(Ztr_t.float().cpu().numpy()).astype(np.float64)
                Uts = empirical_pit(Zts_t.float().cpu().numpy()).astype(np.float64)
                Xhat_np = Xhat_t.float().cpu().numpy()

            vine = fit_vine(Utr, cfg.trunc)
            ll, aic = vine_metrics(vine, Uts)
            bic = vine_bic(vine, Uts)

            ts0 = time.perf_counter()
            try:
                vine.simulate(500, seeds=[seed])
                result["t_sample"] = time.perf_counter() - ts0
            except Exception:
                pass

            tau_rec = kendall_matrix(Xhat_np)
            corr_rec = np.corrcoef(Xhat_np.T)
            result.update(
                ll=ll, nll=-ll, aic=aic, bic=bic, n_params=vine_nparams(vine),
                dep_rec=np.linalg.norm(tau_orig - tau_rec, "fro"),
                tau_err=np.abs(tau_orig - tau_rec)[np.triu_indices(d, 1)].mean(),
                rmse_corr=np.sqrt(np.mean((corr_orig - corr_rec) ** 2))
            )
            del model

        # --- InfoVAE-Vine ---
        elif name == "InfoVAE-Vine":
            model = train_infovae(X_tr, X_vl, d_lat, cfg, seed)
            result["t_train"] = time.perf_counter() - t0

            model.eval()
            Xtr_t = torch.tensor(X_tr, dtype=torch.float32).to(DEVICE)
            Xts_t = torch.tensor(X_ts, dtype=torch.float32).to(DEVICE)

            with torch.no_grad():
                _, _, mu_tr, _ = model(Xtr_t)
                _, _, mu_ts, _ = model(Xts_t)
                Xhat_t = model.decoder(mu_ts)
                Utr = empirical_pit(mu_tr.float().cpu().numpy()).astype(np.float64)
                Uts = empirical_pit(mu_ts.float().cpu().numpy()).astype(np.float64)
                Xhat_np = Xhat_t.float().cpu().numpy()

            vine = fit_vine(Utr, cfg.trunc)
            ll, aic = vine_metrics(vine, Uts)
            bic = vine_bic(vine, Uts)

            ts0 = time.perf_counter()
            try:
                vine.simulate(500, seeds=[seed])
                result["t_sample"] = time.perf_counter() - ts0
            except Exception:
                pass

            tau_rec = kendall_matrix(Xhat_np)
            corr_rec = np.corrcoef(Xhat_np.T)
            result.update(
                ll=ll, nll=-ll, aic=aic, bic=bic, n_params=vine_nparams(vine),
                dep_rec=np.linalg.norm(tau_orig - tau_rec, "fro"),
                tau_err=np.abs(tau_orig - tau_rec)[np.triu_indices(d, 1)].mean(),
                rmse_corr=np.sqrt(np.mean((corr_orig - corr_rec) ** 2))
            )
            del model

        # --- AE-Vine-Selected ---
        elif name == "AE-Vine-Selected":
            model, vine = train_ae_selected(X_tr, X_vl, d_lat, cfg, seed, verbose=False)
            result["t_train"] = time.perf_counter() - t0
            if vine is None:
                return result

            Xts_t = torch.tensor(X_ts, dtype=torch.float32).to(DEVICE)
            model.eval()
            with torch.no_grad():
                Zts, Xhat = model(Xts_t)
                Uts = empirical_pit(Zts.float().cpu().numpy()).astype(np.float64)
                Xhat_np = Xhat.cpu().numpy()

            ll, aic = vine_metrics(vine, Uts)
            bic = vine_bic(vine, Uts)

            ts0 = time.perf_counter()
            try:
                vine.simulate(500, seeds=[seed])
                result["t_sample"] = time.perf_counter() - ts0
            except Exception:
                pass

            tau_rec = kendall_matrix(Xhat_np)
            corr_rec = np.corrcoef(Xhat_np.T)
            result.update(
                ll=ll, nll=-ll, aic=aic, bic=bic, n_params=vine_nparams(vine),
                dep_rec=np.linalg.norm(tau_orig - tau_rec, "fro"),
                tau_err=np.abs(tau_orig - tau_rec)[np.triu_indices(d, 1)].mean(),
                rmse_corr=np.sqrt(np.mean((corr_orig - corr_rec) ** 2))
            )
            del model

        # --- Vine-Direct / Vine-Truncated ---
        elif name in ["Vine-Direct", "Vine-Truncated"]:
            trunc_lvl = 1 if name == "Vine-Truncated" else cfg.trunc
            U_tr_emp = empirical_pit(X_tr)
            vine = fit_vine(U_tr_emp, trunc_lvl)
            result["t_train"] = time.perf_counter() - t0

            ll, aic = vine_metrics(vine, U_ts_emp)
            bic = vine_bic(vine, U_ts_emp)

            ts0 = time.perf_counter()
            try:
                vine.simulate(500, seeds=[seed])
                result["t_sample"] = time.perf_counter() - ts0
            except Exception:
                pass

            result.update(
                ll=ll, nll=-ll, aic=aic, bic=bic, n_params=vine_nparams(vine)
            )

    except Exception as e:
        print(f"  [{name}] FAILED: {e}")
        traceback.print_exc()

    result["mem_mb"] = _mem_mb() - mem0
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return result


# %% 9.4 Benchmark Runner
def run_benchmark(cfg=CFG, methods=None, scenarios=None, seeds=None, auto_d_lat=True):
    """Run the full benchmark for all methods and scenarios."""
    methods = methods or METHODS
    scenarios = scenarios or SCENARIOS
    seeds = seeds or cfg.seeds

    rows = []
    for sc_key, sc in scenarios.items():
        print(f"\n{'=' * 65}\n  {sc['label']}\n{'=' * 65}")

        for seed in seeds:
            print(f"\n  ── seed={seed} ──")
            ds = sc["split_fn"](sc["builder"](seed))
            d_lat_used = sc["d_lat"] if not auto_d_lat else select_d_lat(
                ds["X_train"], cfg.d_lat_var_target
            )
            print(f"    d_lat = {d_lat_used} ({'auto' if auto_d_lat else 'manual'})")

            for meth in methods:
                print(f"\n  [{meth}]")
                r = run_method(meth, ds["X_train"], ds["X_val"], ds["X_test"], d_lat_used, cfg, seed)
                r.update(
                    scenario=sc_key,
                    scenario_label=sc["label"],
                    n_train=len(ds["X_train"]),
                    d=ds["X_train"].shape[1]
                )
                rows.append(r)
                print(f"    LL={r['ll']:+.4f}  AIC={r['aic']:.2f}  DepRec={r.get('dep_rec', np.nan):.4f}  t={r['t_train']:.1f}s")

    return pd.DataFrame(rows)


# =============================================================================
# SECTION 10: STATISTICAL TESTS
# =============================================================================

def run_stats_tests(df: pd.DataFrame) -> Dict:
    """Run Friedman and Wilcoxon statistical tests."""
    results = {}
    wilcoxon_rows = []

    for sc in df["scenario"].unique():
        sub = df[df["scenario"] == sc]
        ls_aic = sub[sub["method"] == "LS-Vine"]["aic"].dropna().values

        for meth in METHODS:
            if meth == "LS-Vine":
                continue
            other_aic = sub[sub["method"] == meth]["aic"].dropna().values
            n = min(len(ls_aic), len(other_aic))
            if n < 2:
                continue

            try:
                stat, p = wilcoxon(ls_aic[:n], other_aic[:n])
                wilcoxon_rows.append({
                    "scenario": sc,
                    "vs": meth,
                    "stat": round(stat, 3),
                    "p": round(p, 4),
                    "sig": "*" if p < 0.05 else ""
                })
            except Exception:
                pass

    results["wilcoxon"] = pd.DataFrame(wilcoxon_rows)

    # Friedman test
    pivot = (df.groupby(["scenario", "seed", "method"])["aic"]
             .mean().reset_index()
             .pivot(index=["scenario", "seed"], columns="method", values="aic")
             .dropna())

    if len(pivot) >= 3 and pivot.shape[1] >= 3:
        try:
            groups = [pivot[m].values for m in pivot.columns]
            stat_f, p_f = friedmanchisquare(*groups)
            results["friedman"] = {"stat": round(stat_f, 3), "p": round(p_f, 4)}
        except Exception as e:
            results["friedman"] = {"stat": np.nan, "p": np.nan, "error": str(e)}

    # Ranking
    rank_df = (df.groupby(["scenario", "seed", "method"])["aic"]
               .mean().reset_index())
    rank_df["rank"] = rank_df.groupby(["scenario", "seed"])["aic"].rank()
    avg_rank = (rank_df.groupby("method")["rank"]
                .mean().sort_values()
                .reset_index()
                .rename(columns={"rank": "mean_rank"}))
    avg_rank["mean_rank"] = avg_rank["mean_rank"].round(3)
    results["ranking"] = avg_rank

    return results


# =============================================================================
# SECTION 11: VISUALIZATION
# =============================================================================

PALETTE = {
    "LS-Vine": "#2a78d6",
    "AE-Vine-Selected": "#8e44ad",
    "PCA-Vine": "#e67e22",
    "ICA-Vine": "#27ae60",
    "FA-Vine": "#9b59b6",
    "KPCA-Vine": "#f39c12",
    "AE-Vine": "#c0392b",
    "VAE-Vine": "#16a085",
    "WAE-Vine": "#d35400",
    "InfoVAE-Vine": "#2980b9",
    "Vine-Direct": "#2c3e50",
    "Vine-Truncated": "#7f8c8d",
}


def fig_boxplots(df: pd.DataFrame):
    """Generate boxplots of AIC per method and scenario."""
    scenarios = df["scenario"].unique()
    fig, axes = plt.subplots(1, len(scenarios), figsize=(5 * len(scenarios), 5), sharey=False)
    if len(scenarios) == 1:
        axes = [axes]

    for ax, sc in zip(axes, scenarios):
        sub = df[df["scenario"] == sc]
        order = [m for m in METHODS if m in sub["method"].unique()]
        sns.boxplot(data=sub, x="method", y="aic", order=order,
                    palette=PALETTE, ax=ax, width=0.6)
        ax.set_title(sc, fontweight="bold", fontsize=11)
        ax.set_xlabel("")
        ax.set_ylabel("Test AIC" if ax == axes[0] else "")
        ax.tick_params(axis="x", rotation=45)
        ax.grid(alpha=0.3, axis="y")

    fig.suptitle("AIC distribution per method (lower = better)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("results/figures/boxplots_aic.png", dpi=130, bbox_inches="tight")
    plt.show()


def fig_violin(df: pd.DataFrame):
    """Generate violin plots of DepRec per method."""
    sub = df[df["dep_rec"].notna() & df["method"].isin(METHODS)]
    fig, ax = plt.subplots(figsize=(12, 5))
    order = [m for m in METHODS if m in sub["method"].unique()]
    sns.violinplot(data=sub, x="method", y="dep_rec", order=order,
                   palette=PALETTE, ax=ax, inner="box")
    ax.set_title("DepRec distribution (lower = better)", fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel("DepRec (Frobenius ‖Δτ‖)")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig("results/figures/violin_deprec.png", dpi=130, bbox_inches="tight")
    plt.show()


def fig_heatmap(df: pd.DataFrame, metric="aic"):
    """Generate heatmap of mean metric per method and scenario."""
    pivot = df.groupby(["method", "scenario"])[metric].mean().unstack()
    fig, ax = plt.subplots(figsize=(max(6, len(pivot.columns) * 1.5), max(4, len(pivot) * 0.8)))
    sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn_r",
                ax=ax, linewidths=0.5, linecolor="white")
    ax.set_title(f"Mean {metric.upper()} per method × scenario", fontweight="bold")
    ax.set_xlabel("Scenario")
    ax.set_ylabel("")
    plt.tight_layout()
    fn = f"results/figures/heatmap_{metric}.png"
    plt.savefig(fn, dpi=130, bbox_inches="tight")
    plt.show()


def fig_timing(df: pd.DataFrame):
    """Generate bar plot of training times."""
    agg = df.groupby("method")["t_train"].mean().reindex(METHODS).dropna()
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = [PALETTE.get(m, "gray") for m in agg.index]
    ax.bar(agg.index, agg.values, color=colors, edgecolor="white")
    ax.set_ylabel("Mean training time (s)")
    ax.set_title("Training time comparison", fontweight="bold")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plt.savefig("results/figures/timing.png", dpi=130, bbox_inches="tight")
    plt.show()


def generate_all_figures(df: pd.DataFrame):
    """Generate all figures."""
    print("\n── Generating figures ──")
    fig_boxplots(df)
    fig_violin(df)
    fig_heatmap(df, "aic")
    fig_heatmap(df, "dep_rec")
    fig_timing(df)


# =============================================================================
# SECTION 12: SUMMARY AND EXPORT
# =============================================================================

def print_summary(df: pd.DataFrame, stats: Dict):
    """Print a summary of benchmark results."""
    print("\n" + "=" * 78)
    print("  BENCHMARK SUMMARY — LS-Vine")
    print("=" * 78)

    agg = (df.groupby(["scenario", "method"])["aic"]
           .mean().reset_index()
           .pivot(index="method", columns="scenario", values="aic")
           .round(1))
    print(agg.to_string())

    print("\n── Mean ranking (AIC, lower = better) ──")
    print(stats["ranking"].to_string(index=False))

    print("\n── Friedman test (overall significance) ──")
    fd = stats.get("friedman", {})
    print(f"  χ² = {fd.get('stat', 'N/A')},  p = {fd.get('p', 'N/A')}")

    print("\n── Wilcoxon: LS-Vine vs each baseline (per scenario) ──")
    if not stats["wilcoxon"].empty:
        print(stats["wilcoxon"].to_string(index=False))

    print("=" * 78)

    n_wins = 0
    for sc in df["scenario"].unique():
        sub = df[df["scenario"] == sc]
        ls = sub[sub["method"] == "LS-Vine"]["aic"].mean()
        for meth in METHODS:
            if meth == "LS-Vine":
                continue
            other = sub[sub["method"] == meth]["aic"].mean()
            if ls < other:
                n_wins += 1
    total = len(df["scenario"].unique()) * (len(METHODS) - 1)
    print(f"  LS-Vine wins: {n_wins}/{total} (method×scenario) pairwise comparisons")
    print("=" * 78)


def generate_latex_tables(df: pd.DataFrame, stats: Dict):
    """Generate LaTeX tables for results."""
    agg = (df.groupby(["scenario", "method"])
           .agg(
               ll=("ll", "mean"),
               aic=("aic", "mean"),
               bic=("bic", "mean"),
               dep_rec=("dep_rec", "mean"),
               tau_err=("tau_err", "mean"),
               rmse_corr=("rmse_corr", "mean"),
               t_train=("t_train", "mean")
           )
           .round(3)
           .reset_index())

    lines = [
        r"\begin{table}[htbp]",
        r"\centering",
        r"\caption{Benchmark results — mean over seeds}",
        r"\label{tab:results}",
        r"\resizebox{\textwidth}{!}{",
        r"\begin{tabular}{llrrrrrr}",
        r"\toprule",
        r"Scenario & Method & LL↑ & AIC↓ & BIC↓ & DepRec↓ & τ-err↓ & RMSE-corr↓ \\",
        r"\midrule"
    ]

    for sc in agg["scenario"].unique():
        sub = agg[agg["scenario"] == sc]
        lines.append(rf"\multicolumn{{8}}{{l}}{{\textit{{{sc}}}}} \\")
        for _, row in sub.iterrows():
            lines.append(
                f"   & {row['method']} & {row['ll']:.3f} & {row['aic']:.1f} "
                f"& {row['bic']:.1f} & {row.get('dep_rec', np.nan):.3f} "
                f"& {row.get('tau_err', np.nan):.3f} "
                f"& {row.get('rmse_corr', np.nan):.3f} \\\\"
            )
        lines.append(r"\midrule")

    lines += [r"\bottomrule", r"\end{tabular}}", r"\end{table}"]

    with open("results/tables/results.tex", "w") as f:
        f.write("\n".join(lines))
    print("Saved: results/tables/results.tex")

    # Ranking table
    rank = stats.get("ranking", pd.DataFrame())
    if not rank.empty:
        r_lines = [
            r"\begin{table}[htbp]",
            r"\centering",
            r"\caption{Mean AIC rank across scenarios}",
            r"\label{tab:ranking}",
            r"\begin{tabular}{lr}",
            r"\toprule",
            r"Method & Mean rank \\ \midrule"
        ]
        for _, row in rank.iterrows():
            r_lines.append(f"{row['method']} & {row['mean_rank']:.3f} \\\\")
        r_lines += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]

        with open("results/tables/ranking.tex", "w") as f:
            f.write("\n".join(r_lines))
        print("Saved: results/tables/ranking.tex")


# =============================================================================
# SECTION 13: MAIN EXECUTION
# =============================================================================

if __name__ == "__main__":
    print("=" * 80)
    print("  🚀 LS-VINE BENCHMARK")
    print("=" * 80)
    print(f"   Methods  : {len(METHODS)}")
    print(f"   Scenarios: {len(SCENARIOS)} ({', '.join(SCENARIOS.keys())})")
    print(f"   Seeds    : {len(CFG.seeds)} (from {CFG.seeds[0]} to {CFG.seeds[-1]})")
    print(f"   Total runs: {len(METHODS) * len(SCENARIOS) * len(CFG.seeds)}")
    print("=" * 80)
    print()

    # Run benchmark
    df_results = run_benchmark(
        cfg=CFG,
        methods=METHODS,
        scenarios=SCENARIOS,
        seeds=CFG.seeds,
        auto_d_lat=True
    )

    # Save results
    df_results.to_csv("results/results_final_benchmark.csv", index=False)
    print("\n✅ Results saved: results/results_final_benchmark.csv")

    # Statistical analysis
    print("\n" + "=" * 80)
    print("  STATISTICAL ANALYSIS")
    print("=" * 80)
    stats_final = run_stats_tests(df_results)

    # Generate figures
    print("\n" + "=" * 80)
    print("  GENERATING FIGURES")
    print("=" * 80)
    generate_all_figures(df_results)

    # Export LaTeX tables
    print("\n" + "=" * 80)
    print("  EXPORTING LATEX TABLES")
    print("=" * 80)
    generate_latex_tables(df_results, stats_final)

    # Final summary
    print("\n" + "=" * 80)
    print("  FINAL SUMMARY")
    print("=" * 80)
    print_summary(df_results, stats_final)

    # Comparative tables
    print("\n" + "=" * 80)
    print("  COMPARATIVE SUMMARY TABLE (means over 10 seeds)")
    print("=" * 80)

    pivot_aic = (df_results.groupby(["scenario", "method"])["aic"]
                 .mean().reset_index()
                 .pivot(index="method", columns="scenario", values="aic")
                 .round(1))
    print("\n📊 Mean AIC by method and scenario (lower = better):")
    print(pivot_aic.to_string())

    pivot_deprec = (df_results.groupby(["scenario", "method"])["dep_rec"]
                    .mean().reset_index()
                    .pivot(index="method", columns="scenario", values="dep_rec")
                    .round(3))
    print("\n📊 Mean DepRec by method and scenario (lower = better):")
    print(pivot_deprec.to_string())

    pivot_time = (df_results.groupby("method")["t_train"]
                  .mean().sort_values()
                  .round(1))
    print("\n📊 Mean training time by method (seconds):")
    print(pivot_time.to_string())

    print("\n" + "=" * 80)
    print("  🎉 BENCHMARK COMPLETED SUCCESSFULLY!")
    print("=" * 80)